# APMS-FW — Chapter 5 corrected reference implementation

Regenerates **all six figures and all five results tables** of the chapter from a single
end-to-end execution, implementing the methodology exactly:

- **C1** Volatility-aware 8-dim feature vector (Eqs. 5.2–5.7), $W_F = 20$, z-scored on burn-in only.
- **C2** GARCH(1,1)-QMLE adaptive monitoring window $(W_{base}, W_{min}, W_{max}) = (30, 10, 60)$; Channel B fixed $W_S = 8$.
- **C3** Fuzzy memberships (volatility-posterior proxy) + membership-weighted reservoirs + **unbiased weighted FW-MMD with the debiased weighted self-term** (Eq. 5.11), bandwidth grid $\{0.5, 1, 2\}\times\hat\sigma_{med}$, stable-regime p90 normalisation, max-aggregated.
- **C4** Two-channel guard: Channel A novelty (Eq. 5.12), Channel B excess-distance ratio (Eq. 5.13), EMA smoothing $(0.85, 0.75)$, rank-combined two-channel score.
- **C5** Per-channel conformal thresholds ($B_{max}=300$, $\alpha_{FAR}=0.10$, ratio floor $0.05$) + asymmetric evidence accumulator $(0.35, 0.88, 0.05)$.
- **C6** Five-level graduated adaptation: L1 soft reservoir update, L2 scale recalibration, L3/L4 **non-destructive refit on regime-matched expanding memory** ($0.999^{\Delta s}$ recency), throttle 25, L4 cooldown 150.
- **C7** Regime-conditional dual HAR-RV head, crisis cap $0.7$, guard $N_{min}=5$.

**Correctness properties whose absence produced the superseded results:** reference updates gated
to confirmed non-drift steps (adaptation never saturates and dies); debiased weighted self-term
(cross-regime distance comparability); leak-free *trailing* persistence (Remark 5.1); realised
targets enter refit memory only after the $h$-step horizon; detection ground truth = **scripted**
switch times with post-onset windows $[s, s+40]$ — never a volatility-derived proxy.

**Outputs** (to `OUTDIR`): `fig1`–`fig6` PNGs, `tab_forecast_synth/real_wf/detection/ablation/frontier.tex`,
and `results_summary.json`. Run top-to-bottom. Whichever run you keep becomes the **single source of
truth**: regenerate figures *and* tables together and update prose numbers from the JSON.

## 0. Run configuration

In [ ]:
# ----------------------- run configuration -----------------------
SEEDS = 20            # detection/ablation seeds (chapter protocol: 20)
OUTDIR = "figures/chapter5"
SKIP_REAL = False     # True -> synthetic experiments only
DATA_DIR = None       # offline CSVs (GSPC.csv, WTI.csv, EURUSD.csv) or None for yfinance
QUICK = False         # True -> 2-seed smoke test, synthetic only

if QUICK:
    SEEDS, SKIP_REAL = 2, True

## 1. Imports and hyperparameter configuration (Table 5.1)

In [ ]:
import json
import os
import sys
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

GREEN, ORANGE, PURPLE, GREY, RED = "#2e8b57", "#e07b39", "#7b2d8b", "#9aa0a6", "#d62728"


# ============================================================================
# Hyperparameters — Table 5.1, identical across all assets and benchmarks.
# ============================================================================
@dataclass
class Config:
    W_F: int = 20                      # feature window (trailing returns)
    W_base: int = 30                   # Channel A adaptive window
    W_min: int = 10
    W_max: int = 60
    W_S: int = 8                       # Channel B short window
    modal_lag: tuple = (30, 5)         # Channel B modal interval [t-30, t-5]
    K: int = 3                         # macro-regimes: 0 calm, 1 volatile, 2 crisis
    bw_grid: tuple = (0.5, 1.0, 2.0)   # x median-heuristic bandwidth
    res_cap: int = 200                 # reservoir FIFO capacity
    res_scored: int = 50               # scored systematic subsample
    lam_A: float = 0.85                # score smoothing
    lam_B: float = 0.75
    alpha_far: float = 0.10            # conformal target
    B_max: int = 300                   # conformal buffer capacity
    tauB_floor: float = 0.05           # Channel B ratio floor
    rho_plus: float = 0.35             # evidence accumulator
    rho_minus: float = 0.88
    e_min: float = 0.05
    refit_throttle: int = 25           # min steps between refits
    recency_decay: float = 0.999       # refit recency weights
    L4_cooldown: int = 150
    crisis_cap: float = 0.7            # dual-head blend cap
    N_min: int = 5                     # crisis-head training guard
    h: int = 5                         # target horizon (next-5-day RV)


CFG = Config()

## 2. Component 1 — volatility-aware feature extraction (Eqs. 5.2–5.7)

In [ ]:
def compute_features(r: np.ndarray, W_F: int = 20) -> np.ndarray:
    """8-dim volatility-aware feature matrix; row t uses r[t-W_F+1 .. t].
    Rows before 2*W_F-2 are NaN (VoV needs a rolling-RV history)."""
    T = len(r)
    F = np.full((T, 8), np.nan)
    # rolling RV series first (needed by VoV)
    rv = np.full(T, np.nan)
    for t in range(W_F - 1, T):
        rv[t] = np.std(r[t - W_F + 1: t + 1], ddof=0)
    for t in range(2 * W_F - 2, T):
        w = r[t - W_F + 1: t + 1]
        mu, sd = w.mean(), w.std(ddof=0)
        sd = max(sd, 1e-12)
        z = (w - mu) / sd
        sk = np.mean(z ** 3)
        ku = np.mean(z ** 4) - 3.0
        var95 = -np.quantile(w, 0.05)
        tail = w[w <= -var95]
        cvar95 = -tail.mean() if len(tail) else var95
        ac1 = np.corrcoef(w[1:], w[:-1])[0, 1] if sd > 1e-10 else 0.0
        if not np.isfinite(ac1):
            ac1 = 0.0
        vov = np.std(rv[t - W_F + 1: t + 1], ddof=0)
        dd = np.max(w) - w[-1]
        F[t] = [rv[t], sk, ku, var95, cvar95, ac1, vov, dd]
    return F

## 3. Component 2 — GARCH(1,1) QMLE and the adaptive monitoring window

In [ ]:
def fit_garch(r: np.ndarray):
    """Quasi-MLE GARCH(1,1) on burn-in returns. Returns (omega, alpha, beta)."""
    r = r - r.mean()
    v = np.var(r)

    def nll(p):
        w, a, b = p
        if w <= 0 or a < 0 or b < 0 or a + b >= 0.999:
            return 1e10
        s2 = np.empty(len(r))
        s2[0] = v
        for t in range(1, len(r)):
            s2[t] = w + a * r[t - 1] ** 2 + b * s2[t - 1]
        s2 = np.maximum(s2, 1e-12)
        return 0.5 * np.sum(np.log(s2) + r ** 2 / s2)

    x0 = np.array([0.05 * v, 0.05, 0.90])
    res = minimize(nll, x0, method="Nelder-Mead",
                   options={"maxiter": 2000, "xatol": 1e-8, "fatol": 1e-8})
    w, a, b = res.x
    if not (w > 0 and a >= 0 and b >= 0 and a + b < 0.999):
        w, a, b = 0.05 * v, 0.05, 0.90
    return float(w), float(a), float(b)

## 4. Memberships — volatility-posterior proxy (Section 5.5.5)
Per-regime Gaussian posterior on log realised volatility, fitted on burn-in labels
(k-means on 20-day rolling volatility, volatility-ordered → calm / volatile / crisis).
Shares the output semantics of the ViFi-HMM bridge; the hard-assignment ablation
one-hots this same vector.

In [ ]:
class MembershipModel:
    """Per-regime Gaussian posterior on log realised volatility, fitted on
    burn-in labels (k-means on 20-day rolling volatility, volatility-ordered).
    Shares the output semantics of the ViFi-HMM bridge: the posterior is
    TEMPERED (T=2) and EMA-SMOOTHED (lam=0.90) so memberships shift smoothly
    through transitions rather than flickering one-hot."""

    TEMP = 2.0
    LAM = 0.90

    def fit(self, log_rv: np.ndarray, labels: np.ndarray, K: int = 3):
        self.K = K
        self.mu = np.zeros(K)
        self.sd = np.ones(K)
        self.pi = np.ones(K) / K
        for k in range(K):
            x = log_rv[labels == k]
            if len(x) >= 2:
                self.mu[k], self.sd[k] = x.mean(), max(x.std(ddof=0), 1e-3)
                self.pi[k] = len(x) / len(log_rv)
        self._u_prev = np.ones(K) / K
        return self

    def posterior(self, log_rv_t: float) -> np.ndarray:
        logp = (np.log(self.pi + 1e-12)
                - np.log(self.sd)
                - 0.5 * ((log_rv_t - self.mu) / self.sd) ** 2)
        logp = (logp - logp.max()) / self.TEMP          # temperature softening
        p = np.exp(logp)
        return p / p.sum()

    def u(self, log_rv_t: float) -> np.ndarray:
        p = self.posterior(log_rv_t)
        self._u_prev = self.LAM * self._u_prev + (1 - self.LAM) * p
        return self._u_prev / self._u_prev.sum()

    def reset(self):
        self._u_prev = np.ones(self.K) / self.K


def burnin_labels(r: np.ndarray, n_burn: int, W_F: int, K: int, seed: int):
    """K-means on 20-day rolling vol over the burn-in, clusters ordered by
    volatility -> 0 calm, 1 volatile, 2 crisis. Also returns rolling vol."""
    rv = pd.Series(r).rolling(W_F).std().values
    ok = np.arange(2 * W_F - 2, n_burn)
    km = KMeans(n_clusters=K, n_init=10, random_state=seed).fit(
        rv[ok].reshape(-1, 1))
    order = np.argsort(km.cluster_centers_.ravel())
    remap = np.empty(K, int)
    remap[order] = np.arange(K)
    labels = np.full(len(r), -1)
    labels[ok] = remap[km.labels_]
    return labels, rv

## 5. Component 3 — FW-MMD (Eq. 5.11, debiased weighted self-term) and soft reservoirs
The debiasing is load-bearing: the biased form carries an $O(1/n_{\mathrm{eff}})$ distortion that
differs across regimes and corrupts the cross-regime comparability on which the best-match
assignment and the Channel-B ratio depend.

In [ ]:
def fwmmd2(X: np.ndarray, Y: np.ndarray, w: np.ndarray,
           bw: float, Kyy_term: float = None) -> float:
    """Unbiased weighted FW-MMD^2 for one bandwidth.
    X: (n,d) window sample; Y: (m,d) scored reference; w: normalised weights.
    Kyy_term may be precomputed (debiased weighted self-term)."""
    n = len(X)
    g = 1.0 / (2.0 * bw * bw)
    Dxx = cdist(X, X, "sqeuclidean")
    Kxx = np.exp(-g * Dxx)
    term1 = (Kxx.sum() - n) / (n * (n - 1)) if n > 1 else 0.0
    Kxy = np.exp(-g * cdist(X, Y, "sqeuclidean"))
    term2 = 2.0 * float(w @ Kxy.mean(axis=0))
    if Kyy_term is None:
        Kyy = np.exp(-g * cdist(Y, Y, "sqeuclidean"))
        sw2 = float(w @ w)
        Kyy_term = (float(w @ Kyy @ w) - sw2) / max(1.0 - sw2, 1e-12)
    return term1 - term2 + Kyy_term


class Reservoir:
    """Per-regime (sample, weight) FIFO reservoir with a systematic scored
    subsample; caches per-bandwidth debiased self-terms."""

    def __init__(self, cap: int, scored: int, bws: np.ndarray):
        self.cap, self.scored, self.bws = cap, scored, bws
        self.X: list = []
        self.w: list = []
        self._dirty = True

    def init_from(self, X: np.ndarray, w: np.ndarray, rng: np.random.Generator):
        keep = w > 1e-4
        X, w = X[keep], w[keep]
        if len(X) > self.cap:
            p = w / w.sum()
            idx = rng.choice(len(X), size=self.cap, replace=False, p=p)
            idx.sort()
            X, w = X[idx], w[idx]
        self.X, self.w = list(X), list(w)
        self._dirty = True

    def append(self, x: np.ndarray, weight: float):
        if weight <= 1e-4:
            return
        self.X.append(x)
        self.w.append(weight)
        if len(self.X) > self.cap:
            self.X.pop(0)
            self.w.pop(0)
        self._dirty = True

    def _rebuild(self):
        X = np.asarray(self.X)
        w = np.asarray(self.w, float)
        m = len(X)
        if m > self.scored:                       # systematic subsample
            idx = np.linspace(0, m - 1, self.scored).round().astype(int)
            X, w = X[idx], w[idx]
        w = w / w.sum()
        self.Ys, self.ws = X, w
        self.self_terms = {}
        sw2 = float(w @ w)
        for bw in self.bws:
            g = 1.0 / (2.0 * bw * bw)
            Kyy = np.exp(-g * cdist(X, X, "sqeuclidean"))
            self.self_terms[bw] = (float(w @ Kyy @ w) - sw2) / max(1 - sw2, 1e-12)
        self._dirty = False

    def scored_view(self):
        if self._dirty:
            self._rebuild()
        return self.Ys, self.ws, self.self_terms


class FWMMDGuard:
    """Aggregated (multi-bandwidth, p90-normalised) FW-MMD distances to the
    K soft references, for one window size class ('A' long / 'B' short)."""

    def __init__(self, bws: np.ndarray, reservoirs: list):
        self.bws, self.res = bws, reservoirs
        self.norm = {bw: 1.0 for bw in bws}      # stable-regime p90 normalisers

    def calibrate(self, windows: list):
        """windows: list of (n_i, d) burn-in window samples (stable regime)."""
        per_bw = {bw: [] for bw in self.bws}
        for X in windows:
            for bw in self.bws:
                vals = []
                for R in self.res:
                    Y, w, st = R.scored_view()
                    vals.append(max(fwmmd2(X, Y, w, bw, st[bw]), 0.0))
                per_bw[bw].append(min(vals))     # best-match distance
        for bw in self.bws:
            v = np.asarray(per_bw[bw])
            self.norm[bw] = max(np.quantile(v, 0.90), 1e-8)

    def distances(self, X: np.ndarray) -> np.ndarray:
        """Aggregated normalised distance to each regime (len K)."""
        out = np.zeros(len(self.res))
        for k, R in enumerate(self.res):
            Y, w, st = R.scored_view()
            best = 0.0
            for bw in self.bws:
                v = max(fwmmd2(X, Y, w, bw, st[bw]), 0.0) / self.norm[bw]
                best = max(best, v)
            out[k] = best
        return out

## 6. Component 7 — HAR predictors and the regime-conditional dual head (Eq. 5.16)

In [ ]:
def har_predictors(r: np.ndarray) -> np.ndarray:
    """h_t = [1, |r|_daily, mean|r|_5, mean|r|_22] (row t causal in r[..t])."""
    a = np.abs(r)
    s = pd.Series(a)
    H = np.column_stack([
        np.ones(len(r)),
        a,
        s.rolling(5).mean().values,
        s.rolling(22).mean().values,
    ])
    return H


def wls(H: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    Wh = H * w[:, None]
    A = Wh.T @ H + 1e-8 * np.eye(H.shape[1])
    return np.linalg.solve(A, Wh.T @ y)


class DualHARHead:
    """theta_s on calm+volatile samples, theta_c on crisis samples;
    smooth crisis blend with cap and training guard (Eq. 5.16)."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.th_s = None
        self.th_c = None
        self.n_c = 0
        self.scale = 1.0

    def fit_initial(self, H, y, labels):
        s_idx = labels <= 1
        c_idx = labels == 2
        w = np.ones(s_idx.sum())
        self.th_s = wls(H[s_idx], y[s_idx], w)
        self.n_c = int(c_idx.sum())
        self.th_c = (wls(H[c_idx], y[c_idx], np.ones(self.n_c))
                     if self.n_c >= self.cfg.N_min else self.th_s.copy())

    def refit(self, memory, regime_of_head: str, cfg: Config, now: int):
        """Non-destructive refit on regime-matched EXPANDING memory with
        exponential recency weights."""
        if regime_of_head == "c":
            rows = [(H, y, s) for (H, y, s, lab) in memory if lab == 2]
        else:
            rows = [(H, y, s) for (H, y, s, lab) in memory if lab <= 1]
        if len(rows) < 30:
            return False
        Hm = np.vstack([h for h, _, _ in rows])
        ym = np.array([y for _, y, _ in rows])
        ds = now - np.array([s for _, _, s in rows])
        w = cfg.recency_decay ** ds
        th = wls(Hm, ym, w)
        if regime_of_head == "c":
            self.th_c = th
            self.n_c = len(rows)
        else:
            self.th_s = th
        self.scale = 1.0
        return True

    def predict(self, h_t: np.ndarray, u_crisis: float, hard: bool) -> float:
        cfg = self.cfg
        if hard:
            uc = 1.0 if (u_crisis >= 0.5 and self.n_c >= cfg.N_min) else 0.0
        else:
            uc = min(u_crisis, cfg.crisis_cap) * (self.n_c >= cfg.N_min)
        yhat = uc * float(h_t @ self.th_c) + (1 - uc) * float(h_t @ self.th_s)
        return max(yhat * self.scale, 1e-8)

## 7. The APMS online loop — Algorithm 5.1
Components 4–6 live inside the loop: two-channel scoring, conformal thresholds, evidence
accumulation, the five-level ladder, non-destructive refits, and the expanding memory.
`adaptation` ∈ {`graduated`, `fixed`, `never`} supports the cost-frontier comparison with
identical base model and refit mechanics.

In [ ]:
@dataclass
class RunResult:
    S_A: np.ndarray
    S_B: np.ndarray
    S_2ch: np.ndarray
    declared: np.ndarray
    levels: np.ndarray
    yhat: np.ndarray
    y: np.ndarray
    valid: np.ndarray
    n_refits: int
    memberships: np.ndarray
    kstar: np.ndarray
    t0: int


def run_apms(r: np.ndarray, n_burn: int, cfg: Config = CFG, seed: int = 0,
             hard: bool = False, adaptation: str = "graduated",
             fixed_interval: int = 0) -> RunResult:
    """adaptation: 'graduated' (APMS), 'fixed' (refit every fixed_interval),
    'never'. Fixed/never keep monitoring outputs but replace the ladder."""
    rng = np.random.default_rng(seed + 10_000)
    T = len(r)
    F = compute_features(r, cfg.W_F)
    feat_start = 2 * cfg.W_F - 2

    # --- burn-in statistics, labels, memberships -------------------------
    labels, rv = burnin_labels(r, n_burn, cfg.W_F, cfg.K, seed)
    ok_b = np.arange(feat_start, n_burn)
    mu_F, sd_F = np.nanmean(F[ok_b], 0), np.nanstd(F[ok_b], 0)
    sd_F = np.where(sd_F < 1e-10, 1.0, sd_F)
    Fz = (F - mu_F) / sd_F
    Fz = np.where(np.isfinite(Fz), Fz, 0.0)

    memb = MembershipModel().fit(np.log(rv[ok_b] + 1e-12), labels[ok_b], cfg.K)

    def u_of(t):
        u = memb.u(np.log(rv[t] + 1e-12))
        if hard:
            v = np.zeros(cfg.K)
            v[int(np.argmax(u))] = 1.0
            return v
        return u

    # --- GARCH window ------------------------------------------------------
    gw, ga, gb = fit_garch(r[:n_burn])
    sig2_base = gw / max(1 - ga - gb, 1e-6)
    sig2 = np.var(r[:n_burn])

    # --- reservoirs ---------------------------------------------------------
    bws0 = float(np.median(cdist(Fz[ok_b][::3], Fz[ok_b][::3])))
    bws = np.array([c * max(bws0, 1e-6) for c in cfg.bw_grid])
    res = [Reservoir(cfg.res_cap, cfg.res_scored, bws) for _ in range(cfg.K)]
    U_b = np.vstack([u_of(t) for t in ok_b])
    for k in range(cfg.K):
        res[k].init_from(Fz[ok_b], U_b[:, k], rng)
    guard = FWMMDGuard(bws, res)

    # stable-regime calibration windows (Channel-A sized) on burn-in
    calib = [Fz[t - cfg.W_base + 1: t + 1]
             for t in range(feat_start + cfg.W_base, n_burn, 7)]
    guard.calibrate(calib[: 80])

    # initial conformal buffers from burn-in Channel-A/B scores
    bufA, bufB = [], []
    for t in range(feat_start + cfg.W_base, n_burn, 7):
        dA = guard.distances(Fz[t - cfg.W_base + 1: t + 1])
        bufA.append(dA.min())
        dS = guard.distances(Fz[t - cfg.W_S + 1: t + 1])
        kS = int(np.argmin(dS))
        bufB.append(max(dS[kS] / max(dS[kS], 1e-8) - 1, 0))   # ~0 on burn-in
        if len(bufA) >= 60:
            break

    # --- forecaster ---------------------------------------------------------
    H = har_predictors(r)
    y = np.full(T, np.nan)                       # next-h-day realised vol
    for t in range(T - cfg.h):
        y[t] = np.std(r[t + 1: t + 1 + cfg.h], ddof=0)
    head = DualHARHead(cfg)
    fit_idx = np.arange(max(feat_start, 22), n_burn - cfg.h)
    head.fit_initial(H[fit_idx], y[fit_idx], labels[fit_idx])

    memory = [(H[t], y[t], t, labels[t]) for t in fit_idx]   # expanding memory

    # --- state ---------------------------------------------------------------
    n_test = T - n_burn
    S_A = np.zeros(n_test)
    S_B = np.zeros(n_test)
    declared = np.zeros(n_test, bool)
    levels = np.zeros(n_test, int)
    yhat = np.full(n_test, np.nan)
    Us = np.zeros((n_test, cfg.K))
    kstars = np.zeros(n_test, int)

    sA_s = sB_s = 0.0
    e = 0.0
    n_drift = 0
    cooldown = 0
    last_refit = -10**9
    n_refits = 0
    kS_hist: list = []
    resid: list = []

    lag_lo, lag_hi = cfg.modal_lag

    for i, t in enumerate(range(n_burn, T)):
        # GARCH update -> adaptive window
        sig2 = gw + ga * r[t - 1] ** 2 + gb * sig2
        W_t = int(np.clip(round(cfg.W_base * np.sqrt(sig2_base / max(sig2, 1e-12))),
                          cfg.W_min, cfg.W_max))
        u = u_of(t)
        Us[i] = u

        # Channel A (long adaptive window)
        XL = Fz[t - W_t + 1: t + 1]
        dA = guard.distances(XL)
        kstar = int(np.argmin(dA))
        kstars[i] = kstar
        sA_raw = float(dA[kstar])

        # Channel B (short window vs recently dominant regime)
        XS = Fz[t - cfg.W_S + 1: t + 1]
        dS = guard.distances(XS)
        kS = int(np.argmin(dS))
        kS_hist.append(kS)
        lo = max(0, len(kS_hist) - 1 - lag_lo)
        hi = max(0, len(kS_hist) - 1 - lag_hi)
        seg = kS_hist[lo: hi + 1]
        m_t = int(np.bincount(seg, minlength=cfg.K).argmax()) if seg else kS
        sB_raw = max(dS[m_t] / max(dS[kS], 1e-8) - 1.0, 0.0)

        # smoothing (cumulative-evidence structure)
        sA_s = cfg.lam_A * sA_s + (1 - cfg.lam_A) * sA_raw
        sB_s = cfg.lam_B * sB_s + (1 - cfg.lam_B) * sB_raw
        S_A[i], S_B[i] = sA_s, sB_s

        # conformal thresholds
        tauA = np.quantile(bufA, 1 - cfg.alpha_far) if len(bufA) >= 30 else np.inf
        tauB = max(np.quantile(bufB, 1 - cfg.alpha_far) if len(bufB) >= 30 else np.inf,
                   cfg.tauB_floor)
        trig = (sA_s > tauA) or (sB_s > tauB)

        # evidence (Eq. 5.14)
        e = e + cfg.rho_plus * (1 - e) if trig else e * cfg.rho_minus
        d_t = (e > cfg.e_min) and trig
        declared[i] = d_t
        n_drift = n_drift + 1 if d_t else 0
        cooldown = max(cooldown - 1, 0)

        # forecast (dual head)
        yhat[i] = head.predict(H[t], u[2], hard)
        if t - cfg.h >= n_burn:                   # track realised residuals
            j = i - cfg.h
            if np.isfinite(yhat[j]) and np.isfinite(y[t - cfg.h]):
                resid.append((yhat[j], y[t - cfg.h]))
                if len(resid) > 40:
                    resid.pop(0)

        # stable-step conformal buffer update
        if kstar == 0 and not trig:
            bufA.append(sA_s)
            bufB.append(sB_s)
            if len(bufA) > cfg.B_max:
                bufA.pop(0)
            if len(bufB) > cfg.B_max:
                bufB.pop(0)

        # ----- adaptation ladder (Eq. 5.15) --------------------------------
        level = 0
        do_refit = False
        if adaptation == "graduated":
            if d_t:
                if e < 0.30:
                    level = 1
                elif e < 0.50:
                    level = 3 if n_drift >= 10 else 2
                else:
                    level = 4 if cooldown == 0 else 3
        elif adaptation == "fixed":
            if fixed_interval > 0 and i > 0 and i % fixed_interval == 0:
                level = 4
        levels[i] = level

        if level == 1:                            # soft reservoir update
            for k in range(cfg.K):
                res[k].append(Fz[t], u[k] if not hard else float(k == np.argmax(u)))
        elif level == 2:                          # scale recalibration
            if len(resid) >= 20:
                num = sum(yy for _, yy in resid)
                den = sum(yh for yh, _ in resid)
                head.scale = float(np.clip(num / max(den, 1e-9), 0.8, 1.25))
        elif level >= 3:                          # non-destructive refit
            if t - last_refit >= cfg.refit_throttle:
                which = "c" if kstar == 2 else "s"
                if head.refit(memory, which, cfg, t):
                    n_refits += 1
                    last_refit = t
                if level == 4:
                    cooldown = cfg.L4_cooldown

        # reference update gated to confirmed non-drift steps
        if adaptation != "graduated" or not d_t:
            for k in range(cfg.K):
                res[k].append(Fz[t], u[k] if not hard else float(k == np.argmax(u)))

        # expanding memory: realised pairs whose horizon has elapsed
        tr = t - cfg.h
        if tr >= n_burn and np.isfinite(y[tr]):
            memory.append((H[tr], y[tr], tr, kstars[i - cfg.h]))

    # two-channel combined score: max of within-series rank transforms
    rA = stats.rankdata(S_A) / len(S_A)
    rB = stats.rankdata(S_B) / len(S_B)
    S2 = np.maximum(rA, rB)

    valid = np.isfinite(y[n_burn:])
    return RunResult(S_A, S_B, S2, declared, levels, yhat, y[n_burn:],
                     valid, n_refits, Us, kstars, n_burn)

## 8. Comparative drift detectors (causal, threshold-free) and AUC

In [ ]:
def det_page_hinkley(r_test: np.ndarray, delta_frac: float = 0.1) -> np.ndarray:
    a = np.abs(r_test)
    delta = delta_frac * a[:50].std()
    mean = 0.0
    U = 0.0
    Umin = 0.0
    out = np.zeros(len(a))
    for t in range(len(a)):
        mean += (a[t] - mean) / (t + 1)
        U += a[t] - mean - delta
        Umin = min(Umin, U)
        out[t] = U - Umin
    return out


def det_adwin_lite(r_test: np.ndarray, delta: float = 0.05,
                   cap: int = 300) -> np.ndarray:
    """ADWIN-style adaptive-window mean-shift statistic on |r_t|.
    Maintains an adaptive window; at each step the score is the maximum over
    split points of |mu1 - mu2| / eps_cut, with the Hoeffding-style cut
    threshold eps_cut = sqrt(ln(4/delta) / (2 m)), m the harmonic mean of the
    sub-window lengths. Score > 1 corresponds to ADWIN's cut condition, upon
    which the older sub-window is dropped (the adaptive shrink)."""
    a = np.abs(r_test)
    win: list = []
    out = np.zeros(len(a))
    ln4d = np.log(4.0 / delta)
    for t in range(len(a)):
        win.append(a[t])
        if len(win) > cap:
            win.pop(0)
        n = len(win)
        if n < 20:
            continue
        w = np.asarray(win)
        cs = np.cumsum(w)
        best, best_c = 0.0, None
        for c in range(8, n - 8, 4):
            mu1 = cs[c - 1] / c
            mu2 = (cs[-1] - cs[c - 1]) / (n - c)
            m = 1.0 / (1.0 / c + 1.0 / (n - c))
            eps = np.sqrt(ln4d / (2.0 * m)) * max(w.std(), 1e-9)
            ratio = abs(mu1 - mu2) / max(eps, 1e-12)
            if ratio > best:
                best, best_c = ratio, c
        out[t] = best
        if best > 1.0 and best_c is not None:      # adaptive shrink
            win = win[best_c:]
    return out


def det_rolling_ks(r_test: np.ndarray, W: int = 30) -> np.ndarray:
    a = np.abs(r_test)
    out = np.zeros(len(a))
    for t in range(2 * W, len(a)):
        s, _ = stats.ks_2samp(a[t - 2 * W: t - W], a[t - W: t])
        out[t] = s
    return out


def auc_score(score: np.ndarray, labels: np.ndarray) -> float:
    """Mann–Whitney AUC of a continuous score against binary labels."""
    n1 = int(labels.sum())
    n0 = len(labels) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    ranks = stats.rankdata(score)
    return float((ranks[labels == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

## 9. Synthetic benchmarks — scripted switch points (Section 5.5.1)
Novelty: 3,000 steps, switches at 750/1500/2250, 60% burn-in. Reassignment: 2,400 steps,
period-200 cycling, 45% burn-in. Ground truth = post-onset windows $[s, s+40]$; the FA mask
keeps only steps $>60$ from any switch.

In [ ]:
def gen_novelty(seed: int):
    """3,000 steps; switches at 750/1500/2250; 60% burn-in. Segment order
    calm -> volatile -> crisis -> calm; the crisis segment straddles the
    burn-in boundary thinly (novelty event at the test-region switch)."""
    rng = np.random.default_rng(seed)
    T = 3000
    params = [(0.01, 0.05, 0.90), (0.02, 0.10, 0.85), (0.05, 0.15, 0.70)]
    seg = np.zeros(T, int)
    seg[750:1500] = 1
    seg[1500:2250] = 2
    seg[2250:] = 0
    r = np.zeros(T)
    s2 = params[0][0] / (1 - params[0][1] - params[0][2])
    for t in range(T):
        w, a, b = params[seg[t]]
        s2 = w + a * (r[t - 1] ** 2 if t else 0.0) + b * s2
        r[t] = np.sqrt(s2) * rng.standard_normal()
    n_burn = int(0.6 * T)
    switches = [s for s in (750, 1500, 2250) if s >= n_burn]
    return r, n_burn, switches, seg


def gen_reassign(seed: int):
    """2,400 steps; regimes cycle with period 200; 45% burn-in — every
    test-region switch is between KNOWN regimes."""
    rng = np.random.default_rng(seed + 500_000)
    T = 2400
    params = [(0.002, 0.05, 0.90), (0.018, 0.10, 0.85), (0.24, 0.15, 0.70)]
    seg = (np.arange(T) // 200) % 3
    r = np.zeros(T)
    s2 = params[0][0] / (1 - params[0][1] - params[0][2])
    for t in range(T):
        w, a, b = params[seg[t]]
        s2 = w + a * (r[t - 1] ** 2 if t else 0.0) + b * s2
        r[t] = np.sqrt(s2) * rng.standard_normal()
    n_burn = int(0.45 * T)
    switches = [s for s in range(200, T, 200) if s >= n_burn]
    return r, n_burn, switches, seg


def post_onset_labels(n_test: int, t0: int, switches, width: int = 40):
    lab = np.zeros(n_test, int)
    for s in switches:
        lab[max(s - t0, 0): min(s - t0 + width + 1, n_test)] = 1
    return lab


def far_mask(n_test: int, t0: int, switches, dist: int = 60):
    m = np.ones(n_test, bool)
    for s in switches:
        m[max(s - t0 - dist, 0): min(s - t0 + dist + 1, n_test)] = False
    return m

## 10. Leak-free forecasting baselines and the Diebold–Mariano (HLN) test
Includes the **trailing** persistence baseline of Remark 5.1 — the naive $\hat y_t = y_{t-1}$
overlaps the target window and is invalid.

In [ ]:
def dm_hln(e1: np.ndarray, e2: np.ndarray, h: int):
    """Diebold–Mariano on squared-error loss with HLN small-sample
    correction. Negative favours model 1."""
    d = e1 ** 2 - e2 ** 2
    d = d[np.isfinite(d)]
    T = len(d)
    dbar = d.mean()
    g0 = np.var(d, ddof=0)
    s = g0
    for k in range(1, h):
        c = np.mean((d[k:] - dbar) * (d[:-k] - dbar))
        s += 2 * c
    var = max(s, 1e-18) / T
    DM = dbar / np.sqrt(var)
    hln = np.sqrt(max(T + 1 - 2 * h + h * (h - 1) / T, 1e-9) / T)
    DMh = DM * hln
    p = 2 * (1 - stats.t.cdf(abs(DMh), df=T - 1))
    return float(DMh), float(p)


def baseline_forecasts(r: np.ndarray, n_burn: int, cfg: Config,
                       labels: np.ndarray, seed: int):
    """All leak-free comparative forecasters on [n_burn, T-h)."""
    T = len(r)
    h = cfg.h
    y = np.full(T, np.nan)
    for t in range(T - h):
        y[t] = np.std(r[t + 1: t + 1 + h], ddof=0)
    idx = np.arange(n_burn, T - h)

    out = {}
    # trailing persistence (Remark 5.1)
    out["Persistence (trailing)"] = np.array(
        [np.std(r[t - h + 1: t + 1], ddof=0) for t in idx])
    # RiskMetrics EWMA
    lam = 0.94
    s2 = np.var(r[:50])
    ew = np.zeros(T)
    for t in range(T):
        s2 = lam * s2 + (1 - lam) * r[t] ** 2
        ew[t] = np.sqrt(s2)
    out["EWMA(0.94)"] = ew[idx]
    # GARCH(1,1) conditional vol
    gw, ga, gb = fit_garch(r[:n_burn])
    s2 = np.var(r[:n_burn])
    gv = np.zeros(T)
    for t in range(1, T):
        s2 = gw + ga * r[t - 1] ** 2 + gb * s2
        gv[t] = np.sqrt(s2)
    out["GARCH(1,1)"] = gv[idx]
    # HAR-RV
    H = har_predictors(r)
    tr = np.arange(max(2 * cfg.W_F, 22), n_burn - h)
    th = wls(H[tr], y[tr], np.ones(len(tr)))
    out["HAR-RV"] = H[idx] @ th
    # Ridge(F8) / GBM(F8) — finite-mask training, mean-impute prediction
    F = compute_features(r, cfg.W_F)
    fs = 2 * cfg.W_F - 2
    tr2 = np.arange(fs, n_burn - h)
    tr2 = tr2[np.all(np.isfinite(F[tr2]), axis=1) & np.isfinite(y[tr2])]
    mu, sd = F[tr2].mean(0), F[tr2].std(0)
    sd = np.where(sd < 1e-10, 1, sd)
    Fz = (F - mu) / sd
    Fz = np.where(np.isfinite(Fz), Fz, 0.0)     # impute residual NaNs at 0 (=mean)
    rg = Ridge(alpha=1.0).fit(Fz[tr2], y[tr2])
    out["Ridge(F8)"] = rg.predict(Fz[idx])
    gbm = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                    learning_rate=0.05,
                                    random_state=seed).fit(Fz[tr2], y[tr2])
    out["GBM(F8)"] = gbm.predict(Fz[idx])
    return out, y[idx]

## 11. LaTeX table writers

In [ ]:
def fmt_p(p):
    return "$<0.001$" if p < 0.001 else f"${p:.3f}$"


def write_tex(path, body):
    with open(path, "w") as f:
        f.write(body)

## 12. Experiment 1 — synthetic comparative forecasting (Table 5.2, Fig 5.2)

In [ ]:
def exp_forecast_synth(outdir, cfg, seed=0):
    r, n_burn, switches, seg = gen_novelty(seed)
    labels, _ = burnin_labels(r, n_burn, cfg.W_F, cfg.K, seed)
    base, y = baseline_forecasts(r, n_burn, cfg, labels, seed)
    run = run_apms(r, n_burn, cfg, seed=seed, hard=False)
    n_eval = len(y)
    preds = {"APMS-FW (ours)": run.yhat[:n_eval]}
    preds.update(base)
    v = np.isfinite(y)
    for p in preds.values():
        v &= np.isfinite(p[:n_eval])

    rows = []
    e_har = base["HAR-RV"][:n_eval][v] - y[v]
    e_per = base["Persistence (trailing)"][:n_eval][v] - y[v]
    for name, p in preds.items():
        e = p[:n_eval][v] - y[v]
        mse, mae = float(np.mean(e ** 2)), float(np.mean(np.abs(e)))
        dmp, pp = dm_hln(e, e_per, cfg.h) if name != "Persistence (trailing)" else (np.nan, np.nan)
        dmh, ph = dm_hln(e, e_har, cfg.h) if name != "HAR-RV" else (np.nan, np.nan)
        rows.append((name, mse, mae, dmp, pp, dmh, ph))
    rows.sort(key=lambda x: x[1])

    # ---- Table 5.2
    L = [r"\begin{tabular}{lcccc}", r"\toprule",
         r"\textbf{Model} & \textbf{MSE} & \textbf{MAE} & "
         r"\textbf{DM vs.\ Persistence} & \textbf{DM vs.\ HAR-RV} \\", r"\midrule"]
    for n, mse, mae, dmp, pp, dmh, ph in rows:
        c1 = "---" if np.isnan(dmp) else f"${dmp:+.2f}$ ({fmt_p(pp)})"
        c2 = "---" if np.isnan(dmh) else f"${dmh:+.2f}$ ({fmt_p(ph)})"
        nm = r"\textbf{APMS-FW (ours)}" if "APMS" in n else n
        L.append(f"{nm} & {mse:.5f} & {mae:.5f} & {c1} & {c2} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]
    write_tex(os.path.join(outdir, "tab_forecast_synth.tex"), "\n".join(L))

    # ---- Fig 5.2
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    names = [x[0] for x in rows][::-1]
    mses = [x[1] for x in rows][::-1]
    cols = [GREEN if "APMS" in n else PURPLE if n == "HAR-RV" else GREY
            for n in names]
    ax.barh(names, mses, color=cols)
    ax.set_xlabel(f"Test MSE (next-{cfg.h}-day realised volatility)")
    ax.set_title("Forecasting comparison — APMS-FW vs industry baselines")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "fig1_forecasting_comparison.png"), dpi=200)
    plt.close(fig)
    return {n: {"MSE": m, "MAE": a} for n, m, a, *_ in rows}, run, r, n_burn, switches

## 13. Experiment 2 — 20-seed detection + exact paired ablation (Tables 5.4/5.5, Figs 5.4/5.5/5.6)
For each (benchmark, seed): APMS-FW and its exact hard-assignment counterpart run on
identical data; paired Wilcoxon signed-rank tests; FA = per-step declaration rate on
segments $>60$ steps from any switch.

In [ ]:
def exp_detection(outdir, cfg, n_seeds, trace_run=None,
                  trace_r=None, trace_nburn=None, trace_switches=None):
    detectors = ["Page–Hinkley", "APMS-FW Channel A (novelty)",
                 "APMS-FW two-channel", "ADWIN-lite", "Rolling KS",
                 "APMS-FW Channel B (switch)"]
    res = {b: {d: [] for d in detectors} for b in ("novelty", "reassignment")}
    abl = {b: {"aucA_fw": [], "aucA_hd": [], "mse_fw": [], "mse_hd": [],
               "fa_fw": [], "fa_hd": []} for b in ("novelty", "reassignment")}

    for seed in range(n_seeds):
        for bname, gen in (("novelty", gen_novelty),
                           ("reassignment", gen_reassign)):
            r, n_burn, switches, seg = gen(seed)
            n_test = len(r) - n_burn
            lab = post_onset_labels(n_test, n_burn, switches)
            fm = far_mask(n_test, n_burn, switches)

            run_fw = run_apms(r, n_burn, cfg, seed=seed, hard=False)
            run_hd = run_apms(r, n_burn, cfg, seed=seed, hard=True)

            rt = r[n_burn:]
            res[bname]["Page–Hinkley"].append(auc_score(det_page_hinkley(rt), lab))
            res[bname]["ADWIN-lite"].append(auc_score(det_adwin_lite(rt), lab))
            res[bname]["Rolling KS"].append(auc_score(det_rolling_ks(rt), lab))
            res[bname]["APMS-FW Channel A (novelty)"].append(
                auc_score(run_fw.S_A, lab))
            res[bname]["APMS-FW Channel B (switch)"].append(
                auc_score(run_fw.S_B, lab))
            res[bname]["APMS-FW two-channel"].append(
                auc_score(run_fw.S_2ch, lab))

            abl[bname]["aucA_fw"].append(auc_score(run_fw.S_A, lab))
            abl[bname]["aucA_hd"].append(auc_score(run_hd.S_A, lab))
            for tag, rr in (("fw", run_fw), ("hd", run_hd)):
                v = rr.valid & np.isfinite(rr.yhat)
                abl[bname][f"mse_{tag}"].append(
                    float(np.mean((rr.yhat[v] - rr.y[v]) ** 2)))
                abl[bname][f"fa_{tag}"].append(float(rr.declared[fm].mean()))
        print(f"  seed {seed + 1}/{n_seeds} done", flush=True)

    # ---- Table 5.4 + Fig 5.4
    L = [r"\begin{tabular}{lcc}", r"\toprule",
         r"\textbf{Detector} & \textbf{Novelty benchmark} & "
         r"\textbf{Reassignment benchmark} \\", r"\midrule"]
    means = {b: {d: (np.nanmean(res[b][d]), np.nanstd(res[b][d]))
                 for d in detectors} for b in res}
    order = sorted(detectors, key=lambda d: -means["novelty"][d][0])
    bestN = max(order, key=lambda d: means["novelty"][d][0])
    bestR = max(order, key=lambda d: means["reassignment"][d][0])
    for d in order:
        mN, sN = means["novelty"][d]
        mR, sR = means["reassignment"][d]
        cN = f"${mN:.3f} \\pm {sN:.3f}$"
        cR = f"${mR:.3f} \\pm {sR:.3f}$"
        if d == bestN:
            cN = f"$\\mathbf{{{mN:.3f} \\pm {sN:.3f}}}$"
        if d == bestR:
            cR = f"$\\mathbf{{{mR:.3f} \\pm {sR:.3f}}}$"
        L.append(f"{d.replace('–','--')} & {cN} & {cR} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]
    write_tex(os.path.join(outdir, "tab_detection.tex"), "\n".join(L))

    fig, ax = plt.subplots(figsize=(9, 4.8))
    x = np.arange(len(order))
    wdt = 0.36
    for off, b, col in ((-wdt / 2, "novelty", GREEN),
                        (wdt / 2, "reassignment", ORANGE)):
        m = [means[b][d][0] for d in order]
        s = [means[b][d][1] for d in order]
        ax.bar(x + off, m, wdt, yerr=s, capsize=3, color=col,
               label=f"{b[:5]}-regime" if b == "novelty" else "recurring-regime")
    ax.axhline(0.5, ls="--", c="k", lw=1)
    ax.set_xticks(x)
    ax.set_xticklabels([d.replace("APMS-FW ", "APMS-FW\n") for d in order],
                       rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("AUC")
    ax.set_title("Detection AUC by method and benchmark")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "fig2_detection_comparison.png"), dpi=200)
    plt.close(fig)

    # ---- Table 5.5 + Fig 5.6 (paired ablation)
    def wilc(a, b):
        a, b = np.asarray(a), np.asarray(b)
        ok = np.isfinite(a) & np.isfinite(b)
        if ok.sum() < 5 or np.allclose(a[ok], b[ok]):
            return np.nan, 0
        st = stats.wilcoxon(a[ok], b[ok])
        return float(st.pvalue), int((a[ok] > b[ok]).sum())

    L = [r"\begin{tabular}{llcccc}", r"\toprule",
         r"\textbf{Benchmark} & \textbf{Quantity} & \textbf{FW-MMD} & "
         r"\textbf{Hard} & \textbf{Wins} & \textbf{Wilcoxon $p$} \\",
         r"\midrule"]
    abl_out = {}
    for b in ("novelty", "reassignment"):
        pA, wA = wilc(abl[b]["aucA_fw"], abl[b]["aucA_hd"])
        pM, _ = wilc(abl[b]["mse_hd"], abl[b]["mse_fw"])   # lower MSE wins
        wM = int(np.sum(np.asarray(abl[b]["mse_fw"]) < np.asarray(abl[b]["mse_hd"])))
        dM = float(np.mean(np.asarray(abl[b]["mse_fw"])
                           - np.asarray(abl[b]["mse_hd"])))
        n = len(abl[b]["aucA_fw"])
        row = (f"\\multirow{{3}}{{*}}{{{b.capitalize()}}}\n"
               f" & Channel A AUC & {np.nanmean(abl[b]['aucA_fw']):.3f} & "
               f"{np.nanmean(abl[b]['aucA_hd']):.3f} & {wA}/{n} & "
               f"{'---' if np.isnan(pA) else f'{pA:.4f}'} \\\\\n"
               f" & Forecast MSE & \\multicolumn{{2}}{{c}}{{mean diff.\\ "
               f"${dM:.2e}$}} & {wM}/{n} & "
               f"{'---' if np.isnan(pM) else f'{pM:.4f}'} \\\\\n"
               f" & FA rate & {np.mean(abl[b]['fa_fw']):.3f} & "
               f"{np.mean(abl[b]['fa_hd']):.3f} & --- & --- \\\\")
        L.append(row)
        if b == "novelty":
            L.append(r"\midrule")
        abl_out[b] = {"aucA_p": pA, "aucA_wins": f"{wA}/{n}",
                      "mse_p": pM, "mse_wins": f"{wM}/{n}",
                      "mse_mean_diff": dM,
                      "fa_fw": float(np.mean(abl[b]["fa_fw"])),
                      "fa_hd": float(np.mean(abl[b]["fa_hd"]))}
    L += [r"\bottomrule", r"\end{tabular}"]
    write_tex(os.path.join(outdir, "tab_ablation.tex"), "\n".join(L))

    fig, ax = plt.subplots(figsize=(6.4, 5.6))
    ax.plot([0.3, 1.0], [0.3, 1.0], "k--", lw=1)
    ax.scatter(abl["novelty"]["aucA_hd"], abl["novelty"]["aucA_fw"],
               c=GREEN, s=55, label="novel")
    ax.scatter(abl["reassignment"]["aucA_hd"], abl["reassignment"]["aucA_fw"],
               c=ORANGE, s=55, label="recurring")
    ax.set_xlabel("Hard-assignment AUC (channel A)")
    ax.set_ylabel("FW-MMD AUC (channel A)")
    ax.set_title("FW-MMD vs hard assignment — points above the line favour FW-MMD")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "fig3_fwmmd_ablation.png"), dpi=200)
    plt.close(fig)

    # ---- Fig 5.5 (seed-0 traces on the novelty benchmark)
    if trace_run is not None:
        run, r, n_burn, switches = trace_run, trace_r, trace_nburn, trace_switches
        t = np.arange(n_burn, len(r))
        fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
        axes[0].plot(t, np.abs(r[n_burn:]), c="0.4", lw=0.6)
        axes[0].set_ylabel("|return|")
        axes[1].plot(t, run.memberships[:, 0], label=r"$u_t$(calm)")
        axes[1].plot(t, run.memberships[:, 1], label=r"$u_t$(volatile)", c=ORANGE)
        axes[1].plot(t, run.memberships[:, 2], label=r"$u_t$(crisis)", c=GREEN)
        axes[1].set_ylabel("membership")
        axes[1].legend(ncol=3, fontsize=8)
        axes[2].plot(t, run.S_A, c="seagreen", label="FW-MMD novelty (ch. A)")
        axes[2].set_ylabel("score")
        axes[2].set_xlabel("time step")
        axes[2].legend(fontsize=8)
        for ax in axes:
            for s in switches:
                ax.axvline(s, ls=":", c="crimson", lw=1.2)
        axes[0].set_title("Novel-regime benchmark (seed 0)")
        fig.tight_layout()
        fig.savefig(os.path.join(outdir, "fig4_monitoring_traces.png"), dpi=200)
        plt.close(fig)

    tab54 = {b: {d: [float(means[b][d][0]), float(means[b][d][1])]
                 for d in detectors} for b in res}
    return tab54, abl_out

## 14. Experiment 3 — adaptation cost frontier (Table 5.6, Fig 5.7)

In [ ]:
def exp_frontier(outdir, cfg, seeds=10):
    """Multi-seed frontier: mean test MSE and mean refit count per policy.
    A single seed makes the policy ordering a draw of noise at these margins
    (0.2–1.5% of MSE); averaging over seeds gives a stable point estimate."""
    policies = ([("Never retrain", "never", 0)]
                + [(f"Fixed every {n} steps", "fixed", n) for n in (200, 100, 50, 25)]
                + [("APMS graduated (ours)", "graduated", 0)])
    acc = {name: {"mse": [], "refits": []} for name, _, _ in policies}

    for seed in range(seeds):
        r, n_burn, switches, _ = gen_novelty(seed)
        for name, mode, n in policies:
            rn = run_apms(r, n_burn, cfg, seed=seed, adaptation=mode,
                          fixed_interval=n)
            v = rn.valid & np.isfinite(rn.yhat)
            acc[name]["mse"].append(float(np.mean((rn.yhat[v] - rn.y[v]) ** 2)))
            acc[name]["refits"].append(rn.n_refits)
        print(f"  frontier seed {seed + 1}/{seeds} done", flush=True)

    rows = [(name,
             float(np.mean(acc[name]["mse"])),
             float(np.mean(acc[name]["refits"])),
             float(np.std(acc[name]["mse"])))
            for name, _, _ in policies]

    # paired per-seed tests: is APMS's MSE distinguishable from each policy?
    ap_mse = np.asarray(acc["APMS graduated (ours)"]["mse"])
    paired = {}
    for name, _, _ in policies:
        if "APMS" in name:
            continue
        other = np.asarray(acc[name]["mse"])
        if len(other) >= 6 and not np.allclose(other, ap_mse):
            w = stats.wilcoxon(ap_mse, other)
            paired[name] = {"p": float(w.pvalue),
                            "apms_wins": int((ap_mse < other).sum()),
                            "n": len(other),
                            "mean_diff": float(np.mean(ap_mse - other))}

    L = [r"\begin{tabular}{lcc}", r"\toprule",
         r"\textbf{Policy} & \textbf{Test MSE} & \textbf{\# refit events} \\",
         r"\midrule"]
    for name, mse, k, _ in rows:
        kk = f"{k:.0f}"
        nm = (r"\textbf{APMS graduated (ours)}" if "APMS" in name else name)
        vv = (f"\\textbf{{{mse:.5f}}} & \\textbf{{{kk}}}" if "APMS" in name
              else f"{mse:.5f} & {kk}")
        L.append(f"{nm} & {vv} \\\\")
    L += [r"\bottomrule", r"\end{tabular}",
          f"% mean over {seeds} seeds; per-policy MSE std "
          + ", ".join(f"{n}: {s:.5f}" for n, _, _, s in rows),
          "% paired per-seed Wilcoxon, APMS vs each policy: "
          + "; ".join(f"{n}: p={v['p']:.3f}, APMS wins {v['apms_wins']}/{v['n']}"
                      for n, v in paired.items())]
    write_tex(os.path.join(outdir, "tab_frontier.tex"), "\n".join(L))

    fig, ax = plt.subplots(figsize=(7, 4.2))
    fx = [k for n, m, k, s in rows if "APMS" not in n]
    fy = [m for n, m, k, s in rows if "APMS" not in n]
    o = np.argsort(fx)
    ax.plot(np.array(fx)[o], np.array(fy)[o], "-o", c="0.45",
            label="fixed schedules")
    ap = [x for x in rows if "APMS" in x[0]][0]
    ax.scatter([ap[2]], [ap[1]], s=110, c=GREEN, zorder=5,
               label="APMS graduated (ours)")
    ax.set_xlabel("# refit events")
    ax.set_ylabel(f"Test MSE (mean over {seeds} seeds)")
    ax.set_title("Cost frontier: accuracy vs adaptation cost")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "fig5_cost_frontier.png"), dpi=200)
    plt.close(fig)
    out = {n: {"mse": m, "refits": k, "mse_std": s} for n, m, k, s in rows}
    out["paired_wilcoxon_apms_vs"] = paired
    return out

## 15. Experiment 4 — real-asset walk-forward (Table 5.3, Fig 5.3)
Expanding-origin, four folds per asset (initial fraction 0.4), per-step errors pooled across
folds, DM vs HAR-RV. Needs `yfinance` or `DATA_DIR` CSVs. Detection metrics are **not**
reported on real assets (no non-circular ground truth).

In [ ]:
def load_real(data_dir):
    specs = {"S&P 500": ("^GSPC", "GSPC.csv", "1990-01-01", "2023-12-31"),
             "WTI": ("CL=F", "WTI.csv", "2000-08-01", "2023-12-31"),
             "EUR/USD": ("EURUSD=X", "EURUSD.csv", "2003-12-01", "2023-12-31")}
    out = {}
    for name, (tick, csv, a, b) in specs.items():
        r = None
        if data_dir and os.path.exists(os.path.join(data_dir, csv)):
            px = pd.read_csv(os.path.join(data_dir, csv))
            col = "Close" if "Close" in px.columns else px.columns[-1]
            p = pd.to_numeric(px[col], errors="coerce").values
            r = _clean_prices(p)
        else:
            try:
                import yfinance as yf
                p = yf.download(tick, start=a, end=b,
                                progress=False)["Close"].values
                r = _clean_prices(p)
            except Exception as e:
                print(f"  ! could not load {name}: {e}")
        if r is not None:
            r = r[np.isfinite(r)]
        if r is not None and len(r) > 1000:
            out[name] = r
    return out


def _clean_prices(p: np.ndarray) -> np.ndarray:
    """Log-returns from a price series, robust to non-positive prints
    (e.g. WTI April 2020) and missing values."""
    p = np.asarray(p, float).ravel()
    p = p[np.isfinite(p) & (p > 0)]
    return np.diff(np.log(p))


def exp_real_assets(outdir, cfg, data_dir, seed=0):
    data = load_real(data_dir)
    if not data:
        print("  ! no real-asset data available — skipping Table 5.3 / Fig 5.3")
        return None
    all_rows = {}
    fig_vals = {}
    for asset, r in data.items():
        T = len(r)
        bounds = [int(f * T) for f in (0.40, 0.55, 0.70, 0.85, 1.00)]
        pooled = {}
        y_pool = []
        for f in range(4):
            n_burn, end = bounds[f], bounds[f + 1]
            seg = r[:end]
            labels, _ = burnin_labels(seg, n_burn, cfg.W_F, cfg.K, seed)
            base, y = baseline_forecasts(seg, n_burn, cfg, labels, seed)
            run = run_apms(seg, n_burn, cfg, seed=seed, hard=False)
            n_eval = len(y)
            preds = dict(base)
            preds["APMS-FW (ours)"] = run.yhat[:n_eval]
            v = np.isfinite(y)
            for p in preds.values():
                v &= np.isfinite(p[:n_eval])
            for nme, p in preds.items():
                pooled.setdefault(nme, []).append((p[:n_eval][v], y[v]))
            print(f"    {asset} fold {f + 1}/4 done", flush=True)
        rows = []
        eh = np.concatenate([p - yy for p, yy in pooled["HAR-RV"]])
        for nme, chunks in pooled.items():
            e = np.concatenate([p - yy for p, yy in chunks])
            mse, mae = float(np.mean(e ** 2)), float(np.mean(np.abs(e)))
            dm, pv = (np.nan, np.nan) if nme == "HAR-RV" else dm_hln(e, eh, cfg.h)
            rows.append((nme, mse, mae, dm, pv))
        all_rows[asset] = rows
        har = [x for x in rows if x[0] == "HAR-RV"][0]
        ap = [x for x in rows if "APMS" in x[0]][0]
        fig_vals[asset] = 100 * (har[1] - ap[1]) / har[1]

    # ---- Table 5.3
    order = ["Persistence (trailing)", "EWMA(0.94)", "GARCH(1,1)", "HAR-RV",
             "Ridge(F8)", "GBM(F8)", "APMS-FW (ours)"]
    L = [r"\begin{tabular}{llcccc}", r"\toprule",
         r"\textbf{Asset} & \textbf{Model} & \textbf{MSE} & \textbf{MAE} & "
         r"\textbf{DM vs.\ HAR} & $p$ \\", r"\midrule"]
    for ai, (asset, rows) in enumerate(all_rows.items()):
        d = {x[0]: x for x in rows}
        L.append(f"\\multirow{{7}}{{*}}{{{asset.replace('&', '\\&')}}}")
        for nme in order:
            if nme not in d:
                continue
            _, mse, mae, dm, pv = d[nme]
            dmc = "---" if np.isnan(dm) else f"${dm:+.2f}$"
            pc = "---" if np.isnan(pv) else fmt_p(pv).strip("$")
            nm2 = r"\textbf{APMS-FW (ours)}" if "APMS" in nme else nme
            L.append(f" & {nm2} & ${mse:.2e}$ & {mae:.6f} & {dmc} & ${pc}$ \\\\")
        if ai < len(all_rows) - 1:
            L.append(r"\midrule")
    L += [r"\bottomrule", r"\end{tabular}"]
    write_tex(os.path.join(outdir, "tab_real_wf.tex"), "\n".join(L))

    # ---- Fig 5.3
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    names = list(fig_vals)
    vals = [fig_vals[n] for n in names]
    ax.bar(names, vals, color=[GREEN if v > 0 else ORANGE for v in vals])
    ax.axhline(0, c="k", lw=1)
    ax.set_ylabel("MSE improvement vs HAR-RV (%)")
    ax.set_title("APMS-FW vs HAR-RV — walk-forward, per asset")
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "fig6_real_assets_vs_har.png"), dpi=200)
    plt.close(fig)
    return {a: {n: {"MSE": m, "MAE": ma,
                    "DM": None if np.isnan(dm) else dm,
                    "p": None if np.isnan(pv) else pv}
                for n, m, ma, dm, pv in rows} for a, rows in all_rows.items()}

## 16. Execute — all experiments, figures, and tables

In [ ]:
os.makedirs(OUTDIR, exist_ok=True)
summary = {"config": {k: (list(v) if isinstance(v, tuple) else v)
                      for k, v in CFG.__dict__.items()}}

print("[1/4] Synthetic comparative forecasting (Table 5.2, Fig 5.2)…")
tab52, run0, r0, nb0, sw0 = exp_forecast_synth(OUTDIR, CFG, seed=0)
summary["forecast_synth"] = tab52
pd.DataFrame(tab52).T

In [ ]:
print(f"[2/4] Detection + paired ablation over {SEEDS} seeds "
      "(Tables 5.4/5.5, Figs 5.4/5.5/5.6)…")
tab54, tab55 = exp_detection(OUTDIR, CFG, SEEDS,
                             trace_run=run0, trace_r=r0,
                             trace_nburn=nb0, trace_switches=sw0)
summary["detection"] = tab54
summary["ablation"] = tab55
print(json.dumps(tab55, indent=2, default=float))

In [ ]:
print("[3/4] Cost frontier (Table 5.6, Fig 5.7)…")
summary["frontier"] = exp_frontier(OUTDIR, CFG, seeds=max(2, min(20, SEEDS)))
pd.DataFrame(summary["frontier"]).T

In [ ]:
if not SKIP_REAL:
    print("[4/4] Real-asset walk-forward (Table 5.3, Fig 5.3)…")
    rr = exp_real_assets(OUTDIR, CFG, DATA_DIR, seed=0)
    if rr:
        summary["real_assets"] = rr
else:
    print("[4/4] Real assets skipped.")

with open(os.path.join(OUTDIR, "results_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(f"All outputs written to {OUTDIR}/")

## 17. Display the generated figures inline

In [ ]:
from IPython.display import Image, display
for fp in sorted(os.listdir(OUTDIR)):
    if fp.endswith(".png"):
        print(fp)
        display(Image(os.path.join(OUTDIR, fp), width=820))

---
**Updating the chapter.** Copy `tab_*.tex` bodies into the corresponding `table` environments,
copy `fig*.png` into `figures/chapter5/`, and update the prose numbers (DM statistics, win counts,
FA rates, frontier MSE/refits) from `results_summary.json` — figures and tables must come from the
**same** run. Numbers will be statistically equivalent, not bit-identical, to any previous execution.